## Project Data Science Lab 2 

In [1]:
import pandas as pd
import os

In [10]:
# Construct the relative path to the Excel file
file_path1 = os.path.join('..', 'dataset', 'dataset.xlsx')
file_path2 = os.path.join('..', 'dataset', 'enisa_skill_set.xlsx')

# Read a specific sheet into a DataFrame
df1 = pd.read_excel(file_path1, sheet_name='study_program_complete')
df2 = pd.read_excel(file_path1, sheet_name='courses_detail')
df3 = pd.read_excel(file_path2, sheet_name='skill_set')

df1 = df1.dropna(subset=['course id'])

# Display the first few rows of the DataFrame
print(f"\nNumber of rows and columns in study program: {df1.shape}")
print(f"\nNumber of rows and columns in course: {df2.shape}")
print(f"\nNumber of rows and columns in skill set: {df3.shape}")


Number of rows and columns in study program: (36, 7)

Number of rows and columns in course: (736, 3)

Number of rows and columns in skill set: (12, 7)


In [ ]:
study_program = df1[['course id','study program name', 'description']]
courses = df2[['course id','course title','course detail description']]
skill_roles = df3[['profile_title','mission','main_tasks','key_skills','key_knowledge']]

### Annotate the dataset

In [ ]:
import pandas as pd
import spacy
import re
from spacy.tokens import Span

# Load a spaCy model
nlp = spacy.load("en_core_web_sm")

# Define regex patterns for custom entities
skill_pattern = r"\b(Data Science|Machine Learning|Random Forests|Naive Bayes)\b"
technology_pattern = r"\b(Python|SQL)\b"
methodology_pattern = r"\b(Feature Selection|Random Forests)\b"

# Function to perform NER and annotation on course_detail
def annotate_course_detail(course_detail):
    # Apply regex to find entities
    skills = [(match.start(), match.end(), "SKILL") for match in re.finditer(skill_pattern, course_detail)]
    technologies = [(match.start(), match.end(), "TECHNOLOGY") for match in re.finditer(technology_pattern, course_detail)]
    methodologies = [(match.start(), match.end(), "METHODOLOGY") for match in re.finditer(methodology_pattern, course_detail)]

    # Combine all matches
    entities = skills + technologies + methodologies

    # Sort entities by their start position (important for creating non-overlapping spans)
    entities = sorted(entities, key=lambda x: x[0])

    # Create a spaCy doc object
    doc = nlp(course_detail)

    # Create spaCy Span objects and add them to the doc
    spans = [Span(doc, doc.char_span(start, end).start, doc.char_span(start, end).end, label=label) for start, end, label in entities]
    doc.ents = spans  # Assign the spans as the entities for the doc

    # Return the annotated doc
    return doc

# Apply annotation to each course detail in the DataFrame
skill_roles['annotated_key_skills'] = skill_roles['key_skills'].apply(annotate_course_detail)

# Example: Printing the results for the first row
for ent in skill_roles['annotated_key_skills'].iloc[0].ents:
    print(ent.text, ent.label_)
